In [23]:
import csv
import os
import time

import requests

BASE = "http://localhost:3000"
GQL = "http://localhost:3000/api/graphql"
CLIENT_ID = "scraper-demo"
CLIENT_SECRET = "scraper-demo-secret"
API_KEY = "sk_demo_000000000000000000000000000000"
TIMEOUT = 30

In [2]:
r = requests.post(
    f"{BASE}/api/oauth/token",
    data={
        "grant_type": "client_credentials",
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
    },
    timeout=TIMEOUT,
)
r.raise_for_status()
token_resp = r.json()
print({k: (v[:28] + "..." if k == "access_token" else v) for k, v in token_resp.items()})

{'access_token': 'eyJhbGciOiJIUzI1NiJ9.eyJzY29...', 'token_type': 'Bearer', 'expires_in': 3600, 'scope': 'read write'}


In [45]:
class ClienteOAuth:
    def __init__(self, base, gql, client_id, client_secret):
        self.base = base
        self._gql = gql
        self.client_id = client_id
        self.client_secret = client_secret
        self.sesion = requests.Session()
        self.sesion.headers["User-Agent"] = "MineriaWeb-2026-2/1.0"
        self._token = None
        self._expira_en = 0.0

    def _renovar(self):
        r = requests.post(
            f"{self.base}/api/oauth/token",
            data={
                "grant_type": "client_credentials",
                "client_id": self.client_id,
                "client_secret": self.client_secret,
            },
            timeout=TIMEOUT,
        )
        r.raise_for_status()
        d = r.json()
        self._token = d["access_token"]
        # renovamos 60 s antes del vencimiento real, con margen de seguridad
        self._expira_en = time.monotonic() + d["expires_in"] - 60
        print(f"  [token renovado, válido ~{d['expires_in']} s]")

    def _cabecera_auth(self):
        if self._token is None or time.monotonic() >= self._expira_en:
            self._renovar()
        return {"Authorization": f"Bearer {self._token}"}
    
    def _cabecera_api(self):
        return {"x-api-key": API_KEY}

    def get(self, path, **params):
        r = self.sesion.get(
            f"{self.base}{path}",
            headers=self._cabecera_auth(),
            params=params,
            timeout=TIMEOUT,
        )
        r.raise_for_status()
        return r.json()
    
    def gql_oauth(self, query, variables=None):
        return self.gql_header(self._cabecera_auth(), query, variables)
    
    def gql_api(self, query, variables=None):
        return self.gql_header(self._cabecera_api(), query, variables)
    
    def gql_header(self, header, query, variables=None):
        r = self.sesion.post(
            f"{self._gql}",
            json={"query": query, "variables": variables or {}},
            headers=header,
            timeout=TIMEOUT
        );
        r.raise_for_status()
        cuerpo = r.json()
        if "errors" in cuerpo:
            raise RuntimeError(cuerpo["errors"])
        return cuerpo["data"]


api = ClienteOAuth(BASE, GQL, CLIENT_ID, CLIENT_SECRET)
print("Prueba:", api.get("/api/productos", page=1, pageSize=1)["pageInfo"])

  [token renovado, válido ~3600 s]
Prueba: {'page': 1, 'pageSize': 1, 'total': 90, 'totalPages': 90}


In [38]:
QUERY_ORDENES = """
query PaginaOrdenes($page: Int!, $pageSize: Int!) {
  ordenes(page: $page, pageSize: $pageSize) {
    items {
      id numero fecha subTotal igv total
      cliente { id nombre apellidos ciudad pais }
      items {
        productoId cantidad subTotal
        producto { codigo nombre categoria }
      }
    }
    pageInfo { page totalPages total }
  }
}
"""

api.gql_oauth(QUERY_ORDENES, {"page": 1, "pageSize": 3})

{'ordenes': {'items': [{'id': 300,
    'numero': 'OR-00300',
    'fecha': '2026-08-10',
    'subTotal': 59586,
    'igv': 10725.48,
    'total': 70311.48,
    'cliente': {'id': 110,
     'nombre': 'Mateo Sebastian',
     'apellidos': 'Dominguez Silva',
     'ciudad': 'Ciudad de Mexico',
     'pais': 'Mexico'},
    'items': [{'productoId': 67,
      'cantidad': 4,
      'subTotal': 27996,
      'producto': {'codigo': 'PROD-067',
       'nombre': 'DJI Air 3',
       'categoria': 'Fotografia y Video'}},
     {'productoId': 85,
      'cantidad': 1,
      'subTotal': 399,
      'producto': {'codigo': 'PROD-085',
       'nombre': 'Samsung T7 SSD Portatil 1TB',
       'categoria': 'Accesorios y Componentes'}},
     {'productoId': 13,
      'cantidad': 2,
      'subTotal': 12998,
      'producto': {'codigo': 'PROD-013',
       'nombre': 'Dell XPS 13 9340 Intel Core i7',
       'categoria': 'Laptops y Computadoras'}},
     {'productoId': 31,
      'cantidad': 3,
      'subTotal': 14997,
      '

In [46]:
def todas_las_ordenes(page_size=50):
    items = []
    page = 1
    while True:
        d = api.gql_oauth(QUERY_ORDENES, {"page": page, "pageSize": page_size})["ordenes"]
        items.extend(d["items"])
        info = d["pageInfo"]
        print(f"  página {info['page']:>2}/{info['totalPages']}  (total {len(items)}/{info['total']})")
        if info["page"] >= info["totalPages"]:
            return items
        page += 1


ordenes = todas_las_ordenes()
print(f"\n{len(ordenes)} órdenes")

  página  1/6  (total 50/300)
  página  2/6  (total 100/300)
  página  3/6  (total 150/300)
  página  4/6  (total 200/300)
  página  5/6  (total 250/300)
  página  6/6  (total 300/300)

300 órdenes


In [47]:
ordenes[1]['id']

299

In [48]:
api.gql_api(QUERY_RESENAS, {"ordenId": ordenes[1]["id"]})

{'comentarios': []}

In [49]:
QUERY_RESENAS = """
query ResenasEntrega($ordenId: Int!) {
  comentarios(tipo: "post_compra", ordenId: $ordenId) {
    id clienteId productoId calificacion texto fecha
    clienteNombre clienteApellidos
  }
}
"""

resenas = []
for o in ordenes:
    for c in api.gql_api(QUERY_RESENAS, {"ordenId": o["id"]})["comentarios"]:
        resenas.append({
            "id": c["id"],
            "orden_id": o["id"],
            "orden_numero": o["numero"],
            "cliente_id": c["clienteId"],
            "cliente_nombre": f"{c['clienteNombre']} {c['clienteApellidos']}",
            "producto_id": c["productoId"],
            "calificacion": c["calificacion"],
            "fecha": c["fecha"],
            "texto": c["texto"],
        })

ordenes_con_resena = len({r["orden_id"] for r in resenas})
print(f"{len(resenas)} reseñas de entrega, repartidas en {ordenes_con_resena} de {len(ordenes)} órdenes")

110 reseñas de entrega, repartidas en 74 de 300 órdenes
